In [2]:
"""
Q-G-CTGAN: Quality-Aware Cluster-Conditioned Oversampling
via Intra-Cluster Synthetic Sample Filtering

Notebook 04a: New Baseline Methods (K-means CTGAN, CTGAN-MOS)

Addresses Reviewer #1 (R1-2) and Reviewer #2 (R2-2): recent GAN-based
oversampling baselines missing from the original comparison.

Environment: venv_sdv (uses the existing sdv/ctgan installation).

Methods implemented here (no official code available; both are
reimplemented from the manuscript description of the original papers):

  - K-means CTGAN (An et al., 2021): confines CTGAN generation to
    k-means-derived cluster regions. Unlike G-CTGAN/Q-G-CTGAN, which
    use GMM (soft, covariance-aware) clustering, K-means CTGAN uses
    hard k-means partitioning as described in the source paper.

  - CTGAN-MOS (Majeed & Hwang, 2023): CTGAN generation followed by a
    "coin-throwing" noise-removal step. NO OFFICIAL IMPLEMENTATION
    EXISTS, and the exact algorithmic details of the coin-throwing
    procedure are only described in the IEEE Access full text (not
    accessible via web search or open repositories at the time of
    this implementation). We reimplement a principled approximation
    consistent with the method's name and abstract-level description:
    each candidate synthetic sample receives an acceptance PROBABILITY
    proportional to its similarity to the real minority distribution
    (estimated via k-NN density in the standardized feature space),
    and is retained via an independent Bernoulli trial ("coin flip")
    at that probability -- a genuinely stochastic accept/reject
    mechanism, as opposed to a deterministic top-k or quantile cutoff
    (which is instead how Q-G-CTGAN's Stage 1 operates, see Notebook
    03). This approximation is disclosed explicitly in the manuscript
    revision as a good-faith reconstruction, not a verified
    reproduction of Majeed & Hwang's exact procedure.
"""

import os
import time
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Path is set once, outside of version-controlled/shared code, to avoid
# exposing local directory structure in the notebook itself.
DATASET_DIR = os.environ.get("QGCTGAN_DATASET_DIR", "./datasets")
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATASET_NAMES = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "yeast_me2", "mammography", "abalone_19", "wine_quality",
    "ecoli", "pageblocks", "protein_homo",
    "satellite", "churn", "secom", "thyroid_sick", "unsw_nb15",
]

CLASSIFIERS = ["RF", "LGBM", "MLP"]

print(f"Datasets    : {len(DATASET_NAMES)}")
print(f"Classifiers : {CLASSIFIERS}")


# ════════════════════════════════════════════════════════════
## 1. Classifiers & Evaluation (same extended metric set as Notebook 03)
# ════════════════════════════════════════════════════════════

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, balanced_accuracy_score, confusion_matrix,
)
from lightgbm import LGBMClassifier


def get_classifier(name, random_state=RANDOM_STATE):
    if name == "RF":
        return RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=3,
            class_weight="balanced", random_state=random_state, n_jobs=-1,
        )
    elif name == "LGBM":
        return LGBMClassifier(
            n_estimators=100, learning_rate=0.05, num_leaves=31,
            class_weight="balanced", random_state=random_state,
            n_jobs=-1, verbose=-1,
        )
    elif name == "MLP":
        return MLPClassifier(
            hidden_layer_sizes=(128, 64), alpha=0.001,
            max_iter=300, random_state=random_state,
        )


def g_mean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return float(np.sqrt(sensitivity * specificity))


def evaluate(model, X_test, y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    return {
        "AUC"         : round(roc_auc_score(y_test, y_prob), 4),
        "PR_AUC"      : round(average_precision_score(y_test, y_prob), 4),
        "F1"          : round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Precision"   : round(precision_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Recall"      : round(recall_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "G_mean"      : round(g_mean_score(y_test, y_pred), 4),
        "Balanced_Acc": round(balanced_accuracy_score(y_test, y_pred), 4),
    }

print("Classifiers & Evaluation ready.")


# ════════════════════════════════════════════════════════════
## 2. K-means CTGAN (An et al., 2021, reimplemented)
# ════════════════════════════════════════════════════════════
# Confines CTGAN generation to k-means-derived cluster regions of the
# minority class. Optimal k selected via silhouette score (the
# manuscript does not specify a cluster-count selection rule for this
# baseline; silhouette score is a standard, defensible default distinct
# from Q-G-CTGAN's BIC-based GMM selection).

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from ctgan import CTGAN


def select_kmeans_k(X_min, k_range=range(2, 11), random_state=RANDOM_STATE):
    best_k, best_score = 2, -1
    for k in k_range:
        if len(X_min) < k * 2:
            break
        try:
            km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
            labels = km.fit_predict(X_min)
            if len(set(labels)) < 2:
                continue
            score = silhouette_score(X_min, labels)
            if score > best_score:
                best_score, best_k = score, k
        except Exception:
            continue
    return best_k


def apply_kmeans_ctgan(X_train, y_train, epochs=100, random_state=RANDOM_STATE):
    X_min = X_train[y_train == 1].astype(float)
    X_maj = X_train[y_train == 0].astype(float)
    n_needed = len(X_maj) - len(X_min)  # target: full balance, Delta = n_maj - n_min

    if n_needed <= 0:
        return X_train, y_train

    best_k = select_kmeans_k(X_min, random_state=random_state)
    km = KMeans(n_clusters=best_k, random_state=random_state, n_init=10)
    labels = km.fit_predict(X_min)

    all_synthetic = []
    per_cluster_target = int(np.ceil(n_needed / best_k))

    for c in range(best_k):
        X_c = X_min[labels == c]
        if len(X_c) < 2:
            continue
        cols = [f"f{i}" for i in range(X_c.shape[1])]
        X_c_df = pd.DataFrame(X_c, columns=cols)
        try:
            model = CTGAN(epochs=epochs, verbose=False)
            model.fit(X_c_df)
            synth = model.sample(per_cluster_target).values.astype(float)
            all_synthetic.append(synth)
        except Exception as e:
            print(f"    K-means CTGAN cluster {c} failed: {e}")

    if not all_synthetic:
        return X_train, y_train

    X_syn = np.vstack(all_synthetic)[:n_needed]
    X_out = np.vstack([X_maj, X_min, X_syn])
    y_out = np.concatenate([
        np.zeros(len(X_maj)), np.ones(len(X_min)), np.ones(len(X_syn))
    ]).astype(int)
    return X_out, y_out

print("K-means CTGAN ready.")


# ════════════════════════════════════════════════════════════
## 3. CTGAN-MOS (Majeed & Hwang, 2023, approximate reimplementation)
# ════════════════════════════════════════════════════════════
# See module docstring for the disclosed approximation of the
# "coin-throwing" noise-removal mechanism: a Bernoulli accept/reject
# trial per candidate, with acceptance probability proportional to
# k-NN density similarity to the real minority class (min-max
# normalised to [0,1] within the candidate pool).

from sklearn.neighbors import NearestNeighbors


def coin_throwing_filter(X_candidates, X_real_minority, n_neighbors=5,
                          random_state=RANDOM_STATE):
    """
    Approximate 'coin-throwing' noise removal: each candidate's
    acceptance probability is proportional to its k-NN density
    similarity to the real minority class (closer candidates get a
    higher probability of survival); actual retention is then decided
    by an independent Bernoulli trial per candidate (the 'coin flip'),
    not a deterministic cutoff.
    """
    rng = np.random.default_rng(random_state)
    nn = NearestNeighbors(n_neighbors=min(n_neighbors, len(X_real_minority)))
    nn.fit(X_real_minority)
    distances, _ = nn.kneighbors(X_candidates)
    mean_dist = distances.mean(axis=1)

    # Convert distance to a similarity-based acceptance probability in [0,1]
    # (closer to real data -> higher probability of being kept)
    inv_dist = 1.0 / (1.0 + mean_dist)
    prob = (inv_dist - inv_dist.min()) / (inv_dist.max() - inv_dist.min() + 1e-8)

    coin_flips = rng.random(len(X_candidates))
    keep_mask = coin_flips < prob
    return X_candidates[keep_mask]


def apply_ctgan_mos(X_train, y_train, epochs=100, candidate_multiplier=3.0,
                     random_state=RANDOM_STATE):
    X_min = X_train[y_train == 1].astype(float)
    X_maj = X_train[y_train == 0].astype(float)
    n_needed = len(X_maj) - len(X_min)

    if n_needed <= 0:
        return X_train, y_train

    cols = [f"f{i}" for i in range(X_min.shape[1])]
    X_min_df = pd.DataFrame(X_min, columns=cols)

    try:
        model = CTGAN(epochs=epochs, verbose=False)
        model.fit(X_min_df)
    except Exception as e:
        print(f"    CTGAN-MOS training failed: {e}")
        return X_train, y_train

    # Over-generate candidates, then apply coin-throwing noise removal
    n_candidates = int(np.round(n_needed * candidate_multiplier))
    candidates = model.sample(n_candidates).values.astype(float)

    survivors = coin_throwing_filter(candidates, X_min, random_state=random_state)

    # If coin-throwing leaves too few survivors, top up with additional
    # generation rounds (bounded retries to avoid infinite loops)
    max_retries = 3
    retry = 0
    while len(survivors) < n_needed and retry < max_retries:
        extra_candidates = model.sample(n_candidates).values.astype(float)
        extra_survivors = coin_throwing_filter(extra_candidates, X_min,
                                                 random_state=random_state + retry + 1)
        survivors = np.vstack([survivors, extra_survivors]) if len(survivors) > 0 else extra_survivors
        retry += 1

    X_syn = survivors[:n_needed] if len(survivors) > n_needed else survivors

    if len(X_syn) == 0:
        return X_train, y_train

    X_out = np.vstack([X_maj, X_min, X_syn])
    y_out = np.concatenate([
        np.zeros(len(X_maj)), np.ones(len(X_min)), np.ones(len(X_syn))
    ]).astype(int)
    return X_out, y_out

print("CTGAN-MOS (approximate reimplementation) ready.")


# ════════════════════════════════════════════════════════════
## 4. Main Experiment Loop
# ════════════════════════════════════════════════════════════
# Same 70/30 stratified split and StandardScaler protocol as Notebooks
# 02 and 03, for direct comparability. Incremental save after each
# dataset (large datasets can take a long time).

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

METHODS = {
    "KMeans_CTGAN": apply_kmeans_ctgan,
    "CTGAN_MOS": apply_ctgan_mos,
}

results = []
out_path = os.path.join(RESULTS_DIR, "04a_new_baselines_results.csv")

for ds_name in DATASET_NAMES:
    path = os.path.join(DATASET_DIR, f"{ds_name}.csv")
    if not os.path.exists(path):
        print(f"[SKIP] {ds_name}: file not found")
        continue

    df = pd.read_csv(path)

    # Cast bool (one-hot encoded) columns to float64 -- same fix applied
    # in Notebooks 02/03 for downstream numeric operations.
    bool_cols = df.select_dtypes(include="bool").columns
    if len(bool_cols):
        df[bool_cols] = df[bool_cols].astype("float64")

    X = df.drop(columns=["target"]).values.astype(float)
    y = df["target"].values.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    print(f"\n{'='*65}")
    print(f"Dataset : {ds_name}  |  n_train={len(X_train):,}  |  minority={y_train.mean():.2%}")
    print(f"{'='*65}")

    for method_name, method_fn in METHODS.items():
        print(f"\n  -- {method_name} --")
        t0 = time.time()
        try:
            X_res, y_res = method_fn(X_train.copy(), y_train.copy())
            gen_time = round(time.time() - t0, 2)
            print(f"  Generated: n={len(y_res):,}  minority={y_res.mean():.2%}  [{gen_time:.1f}s]")
        except Exception as e:
            print(f"  {method_name} failed: {e}")
            continue

        for clf_name in CLASSIFIERS:
            clf = get_classifier(clf_name)
            t1 = time.time()
            try:
                clf.fit(X_res, y_res)
                train_time = round(time.time() - t1, 2)
                metrics = evaluate(clf, X_test, y_test)
            except Exception as e:
                print(f"    {clf_name} - Failed: {e}")
                continue

            results.append({
                "dataset": ds_name, "method": method_name, "classifier": clf_name,
                "generation_time": gen_time, "train_time": train_time,
                "n_synthetic": len(y_res) - len(y_train), **metrics,
            })
            print(f"    {clf_name:<5} | AUC={metrics['AUC']:.4f}  "
                  f"PR-AUC={metrics['PR_AUC']:.4f}  G-mean={metrics['G_mean']:.4f}")

    pd.DataFrame(results).to_csv(out_path, index=False)
    print(f"\n  === {ds_name} complete; results saved incrementally ===")

print("\n04a experiment complete.")

Datasets    : 16
Classifiers : ['RF', 'LGBM', 'MLP']
Classifiers & Evaluation ready.
K-means CTGAN ready.
CTGAN-MOS (approximate reimplementation) ready.

Dataset : credit_default  |  n_train=21,000  |  minority=22.12%

  -- KMeans_CTGAN --
  Generated: n=32,710  minority=50.00%  [78.6s]
    RF    | AUC=0.7609  PR-AUC=0.5409  G-mean=0.5899
    LGBM  | AUC=0.7811  PR-AUC=0.5456  G-mean=0.5852
    MLP   | AUC=0.6799  PR-AUC=0.3657  G-mean=0.5320

  -- CTGAN_MOS --
  Generated: n=32,710  minority=50.00%  [89.9s]
    RF    | AUC=0.7659  PR-AUC=0.5436  G-mean=0.5878
    LGBM  | AUC=0.7810  PR-AUC=0.5519  G-mean=0.5845
    MLP   | AUC=0.6947  PR-AUC=0.3917  G-mean=0.5429

  === credit_default complete; results saved incrementally ===

Dataset : fraud_detection  |  n_train=199,364  |  minority=0.17%

  -- KMeans_CTGAN --
  Generated: n=398,040  minority=50.00%  [64.3s]
    RF    | AUC=0.9579  PR-AUC=0.8019  G-mean=0.9040
    LGBM  | AUC=0.9510  PR-AUC=0.8026  G-mean=0.8889
    MLP   | AUC=0.9

In [3]:
import pandas as pd

results_04a = pd.read_csv("./results/04a_new_baselines_results.csv", keep_default_na=False)

# Confirm the extent of the G-mean=0 issue
zero_gmean = results_04a[results_04a["G_mean"] == 0]
print(f"Rows with G-mean = 0: {len(zero_gmean)} / {len(results_04a)}")
print()
print(zero_gmean[["dataset", "method", "classifier", "AUC", "PR_AUC", "G_mean", "Precision", "Recall"]].to_string(index=False))

Rows with G-mean = 0: 12 / 96

   dataset       method classifier    AUC  PR_AUC  G_mean  Precision  Recall
 yeast_me2    CTGAN_MOS         RF 0.9231  0.2510     0.0     0.4831  0.4965
abalone_19 KMeans_CTGAN         RF 0.7261  0.0172     0.0     0.4960  0.5000
abalone_19 KMeans_CTGAN       LGBM 0.7599  0.0424     0.0     0.4960  0.5000
abalone_19    CTGAN_MOS         RF 0.7313  0.0246     0.0     0.4960  0.5000
abalone_19    CTGAN_MOS       LGBM 0.6930  0.0282     0.0     0.4960  0.5000
abalone_19    CTGAN_MOS        MLP 0.8218  0.0331     0.0     0.4960  0.4972
     secom KMeans_CTGAN         RF 0.7900  0.2298     0.0     0.4671  0.5000
     secom KMeans_CTGAN       LGBM 0.8177  0.2737     0.0     0.4671  0.5000
     secom    CTGAN_MOS         RF 0.7722  0.1780     0.0     0.4671  0.5000
     secom    CTGAN_MOS       LGBM 0.7774  0.1909     0.0     0.4669  0.4966
 unsw_nb15 KMeans_CTGAN         RF 0.9038  0.1375     0.0     0.4950  0.5000
 unsw_nb15    CTGAN_MOS         RF 0.9046  0.

In [4]:
"""
Cross-check: does the G-mean=0 phenomenon also appear in Notebook 02
(non-generative baselines) and Notebook 03 (Q-G-CTGAN) for the same
three problematic datasets (abalone_19, secom, unsw_nb15)?
"""

import pandas as pd

results_02 = pd.read_csv("./results/02_baseline_results_scaled.csv", keep_default_na=False)
results_03 = pd.read_csv("./results/03_qgctgan_results.csv", keep_default_na=False)

# Notebook 02 doesn't have G-mean (only AUC was tracked there originally),
# so we check via a proxy: were G-mean/Balanced_Acc tracked anywhere for it?
print("Columns in 02 results:", list(results_02.columns))
print()
print("Columns in 03 results:", list(results_03.columns))

Columns in 02 results: ['dataset', 'oversampler', 'classifier', 'auc_mean', 'auc_std', 'time_mean_sec', 'time_total_sec']

Columns in 03 results: ['dataset', 'oversampling', 'classifier', 'alpha_mode', 'alpha_value', 'mmd_stage', 'best_k', 'n_candidates', 'candidate_multiplier_used', 'multiplier_capped', 'n_syn_target', 'n_synthetic', 'minority_ratio', 'target_gap', 'mmd_before', 'mmd_after', 'gmm_time', 'ctgan_train_time', 'generation_time', 'stage1_time', 'stage2_time', 'filter_time', 'train_time', 'AUC', 'PR_AUC', 'F1', 'Precision', 'Recall', 'G_mean', 'Balanced_Acc']


In [5]:
print("=" * 90)
print("Q-G-CTGAN (Notebook 03) G-mean for the three problematic datasets")
print("=" * 90)
check_03 = results_03[
    (results_03["dataset"].isin(["abalone_19", "secom", "unsw_nb15"])) &
    (results_03["oversampling"] == "Q-G-CTGAN_adaptive_MMDon")
]
print(check_03[["dataset", "classifier", "AUC", "G_mean", "Precision", "Recall"]].to_string(index=False))

Q-G-CTGAN (Notebook 03) G-mean for the three problematic datasets
   dataset classifier    AUC  G_mean  Precision  Recall
abalone_19         RF 0.7800  0.0000     0.4960  0.5000
abalone_19       LGBM 0.6982  0.0000     0.4960  0.5000
abalone_19        MLP 0.7660  0.3137     0.5202  0.5420
     secom         RF 0.7696  0.0000     0.4671  0.5000
     secom       LGBM 0.8005  0.0000     0.4671  0.5000
     secom        MLP 0.6386  0.2514     0.5594  0.5220
 unsw_nb15         RF 0.9052  0.0000     0.4950  0.5000
 unsw_nb15       LGBM 0.9233  0.3180     0.9229  0.5505
 unsw_nb15        MLP 0.8938  0.2583     0.7114  0.5330
